# Install Dependencies

In [ ]:
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.3/366.3 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 33.0 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets
!pip install -q regex

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 3.3 MB/s eta 0:00:00


In [ ]:
!pip install -q diffusers

# login

In [ ]:
import huggingface_hub
huggingface_hub.login('HF_TOKEN')

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.utils import shuffle
import os
import re
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             precision_score,
                             recall_score,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
import emoji
import pyarabic.araby as araby

In [ ]:
import pandas as pd

In [ ]:
import torch
import torch.distributed as dist

# Load Model

In [ ]:
model_name = "ALLaM-AI/ALLaM-7B-Instruct-preview"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )

# Assign pad_token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.03G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=20,
                temperature=0.2
               )

Device set to use cuda:0


# Zero Shot

In [ ]:
import pandas as pd
data = pd.read_excel('Math-QA-ArabicPrompt-Zero Shot.xlsx')

In [ ]:
data.shape

(200, 1)

# Predict

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(data))):
    prompt = data.iloc[i]["prompt"]
    # prompt = prompt[:max_length]
    # result = pipe(prompt, pad_token_id=pipe.tokenizer.eos_token_id)
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("الإجابة:")[-1].strip()

    pred.append(answer)

100%|██████████| 200/200 [02:30<00:00,  1.33it/s]


In [ ]:
pred_zero = pd.DataFrame()
pred_zero['Predicted'] = pred
pred_zero['Predicted'].value_counts()

,count
Predicted,
د) لاشيء مما سبق,7
أ / ب / ج / د) الإجابة,6
ب,4
ج) 6,3
أ) 2,3
...,...
د) 878,1
ب) 80,1
أ) وتر,1


In [ ]:
nor_pre = []
for pr in pred_zero['Predicted']:
  if "أ)" in pr:
    nor_pre.append("A")
  elif "أ / أ" in pr:
    nor_pre.append("A")
  elif "أ / ب / ج / د)" in pr:
    nor_pre.append("Unclassified")
  elif "ب)" in pr:
    nor_pre.append("B")
  elif "ب" in pr:
    nor_pre.append("B")
  elif "ج)" in pr:
    nor_pre.append("C")
  elif "ج" in pr:
    nor_pre.append("C")
  elif "د)" in pr:
    nor_pre.append("D")
  else:
    nor_pre.append("A")

In [ ]:
pred_zero['Normalized Prediction'] = nor_pre

In [ ]:
pred_zero['Normalized Prediction'].value_counts()

,count
Normalized Prediction,
A,72
C,55
B,51
D,16
Unclassified,6


In [ ]:
pred_zero['prompt'] = data['prompt']
pred_zero.head()

,Predicted,Normalized Prediction,prompt
0,أ) 36,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
1,أ) 23688,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
2,ج) 83,C,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
3,ج) كلاهما,C,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
4,أ) 9+7,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...


In [ ]:
pred_zero.to_excel('Allam Zero Shot QA Math.xlsx', index = False)

In [ ]:
true = pd.read_excel('sample_math.xlsx')
y_true = true['Answer Key'].values
print(classification_report(y_true, pred_zero['Normalized Prediction'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.6389    0.7077    0.6715        65
           B     0.6667    0.7083    0.6869        48
           C     0.7455    0.6119    0.6721        67
           D     0.6250    0.5000    0.5556        20
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.6550       200
   macro avg     0.5352    0.5056    0.5172       200
weighted avg     0.6799    0.6550    0.6638       200



# Pred Few Shot

In [ ]:
data2 = pd.read_excel('Math-QA-ArabicPrompt-Few Shot.xlsx')

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(data2))):
    prompt = data2.iloc[i]["prompt"]
    # prompt = prompt[:max_length]
    # result = pipe(prompt, pad_token_id=pipe.tokenizer.eos_token_id)
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("الإجابة:")[-1].strip()

    pred.append(answer)

100%|██████████| 200/200 [03:11<00:00,  1.04it/s]


In [ ]:
pred_few = pd.DataFrame()
pred_few['Predicted'] = pred
pred_few['Predicted'].value_counts()

,count
Predicted,
د) لاشيء مما سبق,4
ب) 2,4
ج) 6,4
أ) 10,3
أ) 12,2
...,...
ب) 3,1
أ) 8,1
ج) 20,1


In [ ]:
nor_pre = []
for pr in pred_few['Predicted']:
  if "أ)" in pr:
    nor_pre.append("A")
  elif "أ / أ" in pr:
    nor_pre.append("A")
  elif "أ / ب / ج / د)" in pr:
    nor_pre.append("Unclassified")
  elif "ب)" in pr:
    nor_pre.append("B")
  elif "ب" in pr:
    nor_pre.append("B")
  elif "ج)" in pr:
    nor_pre.append("C")
  elif "ج" in pr:
    nor_pre.append("C")
  elif "د)" in pr:
    nor_pre.append("D")

In [ ]:
pred_few['Normalized Prediction'] = nor_pre
pred_few['Normalized Prediction'].value_counts()

,count
Normalized Prediction,
A,67
C,59
B,55
D,19


In [ ]:
pred_few['prompt'] = data2['prompt']
pred_few.head()

,Predicted,Normalized Prediction,prompt
0,أ) 36,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
1,أ) 23688,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
2,ب) 71,B,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
3,ج) كلاهما,C,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
4,أ) 9+7,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...


In [ ]:
pred_few.to_excel('Allam Few Shot QA Biology.xlsx', index = False)

In [ ]:
print(classification_report(y_true, pred_few['Normalized Prediction'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.7313    0.7538    0.7424        65
           B     0.6545    0.7500    0.6990        48
           C     0.7288    0.6418    0.6825        67
           D     0.6316    0.6000    0.6154        20

    accuracy                         0.7000       200
   macro avg     0.6866    0.6864    0.6848       200
weighted avg     0.7021    0.7000    0.6992       200



# CoT

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=200,
                temperature=0.2
               )

Device set to use cuda:0


In [ ]:
data3 = pd.read_excel('Math-QA-ArabicPrompt-CoT.xlsx')

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(data3))):
    prompt = data3.iloc[i]["prompt"]
    # prompt = prompt[:max_length]
    # result = pipe(prompt, pad_token_id=pipe.tokenizer.eos_token_id)
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("السؤال الذي يجب عليك الإجابة عليه:")[-1].strip()

    pred.append(answer)

100%|██████████| 200/200 [07:50<00:00,  2.35s/it]


In [ ]:
pred_cot = pd.DataFrame()
pred_cot['Predicted'] = pred
pred_cot['Predicted'].value_counts()

,count
Predicted,
السؤال:\nاحد الاعداد التالية يقبل القسمة على 6 دون باق .........\n الخيارات:\nأ) 36\nب) 56\nج) 63\nد) لاشيء مما سبق \nالإجابة:\nأ) 36,1
السؤال:\nجد ناتج مايلي : 6 × 3981\n\n الخيارات:\nأ) 23688\nب) 23686\nج) 23886\nد) 32886 \nالإجابة:\nأ) 23688,1
السؤال:\nعدد من منزلتین مجموعهما = ١٠\n الخيارات:\nأ) 73\nب) 71\nج) 83\nد) لاشيء مما سبق \nالإجابة:\nب) 71,1
السؤال:\nمن أدوات القياس\n الخيارات:\nأ) القدم\nب) الذراع\nج) كلاهما\nد) لاشيء مما سبق \nالإجابة:\nج) كلاهما,1
السؤال:\nأي من المسائل التالية ناتج جمعها يساوي 16\n الخيارات:\nأ) 9+7\nب) 10+7\nج) 8+9\nد) 9+6 \nالإجابة:\nأ) 9+7,1
...,...
السؤال:\nكل الاعداد الاولية فردية ما عدا .......\n الخيارات:\nأ) 5\nب) 3\nج) 1\nد) لاشيء مما سبق \nالإجابة:\nب) 3,1
السؤال:\nالعدد الذي یسبق التسعة هو\n الخيارات:\nأ) 8\nب) 5\nج) 1\nد) لاشيء مما سبق \nالإجابة:\nأ) 8,1
السؤال:\nحوط العدد الاكبر\n الخيارات:\nأ) 16\nب) 17\nج) 20\nد) 39 \nالإجابة:\nج) 20,1


In [ ]:
nor_pre = []
for pr in pred_cot['Predicted']:
  if "الإجابة:" in pr:
    answer = pr.split("الإجابة:")[-1].strip()
    if "أ)" in answer:
      nor_pre.append("A")
    elif "أ / أ" in answer:
      nor_pre.append("A")
    elif "أ / ب / ج / د)" in answer:
      nor_pre.append("Unclassified")
    elif "ب)" in answer:
      nor_pre.append("B")
    elif "ب" in answer:
      nor_pre.append("B")
    elif "ج)" in answer:
      nor_pre.append("C")
    elif "ج" in answer:
      nor_pre.append("C")
    elif "د)" in answer:
      nor_pre.append("D")
  elif "الإجابة النهائية:" in pr:
    answer = pr.split("الإجابة النهائية:")[-1].strip()
    if "أ)" in answer:
      nor_pre.append("A")
    elif "أ / أ" in answer:
      nor_pre.append("A")
    elif "أ / ب / ج / د)" in answer:
      nor_pre.append("Unclassified")
    elif "ب)" in answer:
      nor_pre.append("B")
    elif "ب" in answer:
      nor_pre.append("B")
    elif "ج)" in answer:
      nor_pre.append("C")
    elif "ج" in answer:
      nor_pre.append("C")
    elif "د)" in answer:
      nor_pre.append("D")
  elif 'الإجابة الصحيحة هي "لاشيء مما سبق".' in pr:
    nor_pre.append("D")
  elif 'الإجابة الصحيحة هي 17.' in pr:
    nor_pre.append("A")
  elif 'الناتج الصحيح هو 6.' in pr:
    nor_pre.append("C")
  elif 'د) لاشيء مما سبق: غير صحيح لأن الخيار (أ) صحيح.' in pr:
    nor_pre.append("A")
  else:
    print(pr)

In [ ]:
pred_cot['Normalized Prediction'] = nor_pre
pred_cot['Normalized Prediction'].value_counts()

,count
Normalized Prediction,
A,73
B,62
C,61
D,4


In [ ]:
pred_cot['prompt'] = data3['prompt']
pred_cot.head()

,Predicted,Normalized Prediction,prompt
0,السؤال:\nاحد الاعداد التالية يقبل القسمة على 6...,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
1,السؤال:\nجد ناتج مايلي : 6 × 3981\n\n الخيارا...,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
2,السؤال:\nعدد من منزلتین مجموعهما = ١٠\n الخيار...,B,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
3,السؤال:\nمن أدوات القياس\n الخيارات:\nأ) القدم...,C,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
4,السؤال:\nأي من المسائل التالية ناتج جمعها يساو...,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...


In [ ]:
pred_cot.to_excel('Allam CoT QA Math.xlsx', index = False)

In [ ]:
print(classification_report(y_true, pred_cot['Normalized Prediction'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.6301    0.7077    0.6667        65
           B     0.5968    0.7708    0.6727        48
           C     0.7377    0.6716    0.7031        67
           D     0.7500    0.1500    0.2500        20

    accuracy                         0.6550       200
   macro avg     0.6787    0.5750    0.5731       200
weighted avg     0.6702    0.6550    0.6387       200

